# Fixed vs Regularised Neural Portfolio Allocation Comparison

This notebook compares the simple fixed-rule volatility-targeted allocator against the regularised neural Sharpe-trained allocator. Both strategies use the same real final probability file for inference: `deliverables/final_predictions.csv`.

The neural allocator is trained on `deliverables/insample_predicitons/insample_predictions.csv` and then applied to the Jan-Jun 2022 final predictions. The updated neural version includes anti-overfitting controls: validation early stopping, a probability gate, no primary-signal flipping, fewer features, stronger L2 regularisation, and exposure/turnover penalties.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

repo_root = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / 'deliverables' / 'final_predictions.csv').exists()
)
comparison_dir = repo_root / 'deliverables' / 'portfolio_weight_outputs' / 'strategy_comparison'
fixed_dir = repo_root / 'deliverables' / 'portfolio_weight_outputs' / 'fixed_strategy'
neural_dir = repo_root / 'deliverables' / 'portfolio_weight_outputs' / 'neural_strategy'

strategy_summary = pd.read_csv(comparison_dir / 'strategy_summary.csv')
instrument_summary = pd.read_csv(comparison_dir / 'instrument_summary.csv')
daily = pd.read_csv(comparison_dir / 'daily_exposure_and_pnl.csv', parse_dates=['date'])
row_comparison = pd.read_csv(comparison_dir / 'fixed_vs_neural_row_comparison.csv', parse_dates=['date'])

strategy_summary

## Headline Result

The regularised neural strategy now slightly outperforms fixed on annualized Sharpe in the Jan-Jun 2022 next-day PnL proxy. In the regenerated comparison, fixed has annualized Sharpe around `5.05`, while the regularised neural strategy has annualized Sharpe around `5.13`.

The important change is that neural is no longer negative. The anti-overfitting controls made it much more conservative: it now trades the same gated rows as fixed and uses lower gross exposure. Fixed still has higher cumulative PnL, but neural has slightly better risk-adjusted performance because it earns that PnL with lower volatility.

In [ ]:
display(Image(filename=str(comparison_dir / 'annualized_sharpe_by_strategy.png')))

## Comparison 1: Activity And Exposure

The old neural strategy traded every prediction row, which was a major overfitting symptom. The regularised neural strategy now uses the same probability gate as fixed, so both strategies have the same number of active rows.

The main exposure difference is now size, not activity. Neural runs lower mean absolute weight and lower mean daily gross exposure than fixed. That reduces risk and prevents the previous blow-up, while still keeping enough exposure to achieve a slightly higher Sharpe.

In [ ]:
strategy_summary[['strategy', 'nonzero_weights', 'mean_abs_weight', 'mean_daily_gross', 'max_daily_gross', 'mean_active_count']]

In [ ]:
display(Image(filename=str(comparison_dir / 'daily_gross_exposure.png')))
display(Image(filename=str(comparison_dir / 'daily_net_exposure.png')))

## Comparison 2: PnL Path

The cumulative PnL proxy now shows both strategies making money over the 2022 inference period. Fixed still has higher cumulative PnL because it takes larger positions on the accepted rows.

The regularised neural model solved the biggest overfitting failure by no longer flipping signals or trading weak probabilities. Its trade-off is conservative sizing: lower exposure gives lower volatility and slightly higher Sharpe, but also lower cumulative PnL.

In [ ]:
strategy_summary[['strategy', 'mean_daily_pnl', 'annualized_vol', 'annualized_sharpe', 'cumulative_pnl']]

In [ ]:
display(Image(filename=str(comparison_dir / 'cumulative_pnl_proxy.png')))

## Comparison 3: Instrument-Level Behaviour

At instrument level, the neural allocator mainly changes the magnitude of exposure relative to the fixed rule. This is useful here because the learned conviction model reduces exposure while preserving enough profitable signal to improve Sharpe.

The training probabilities are pre-2022, while inference is Jan-Jun 2022. Energy markets in 2022 were unusually volatile, so the updated neural model deliberately avoids aggressive extrapolation. That makes it safer and better on Sharpe, though fixed still earns more cumulative PnL because it carries larger positions.

In [ ]:
instrument_summary

In [ ]:
display(Image(filename=str(comparison_dir / 'mean_abs_weight_by_instrument.png')))
display(Image(filename=str(comparison_dir / 'fixed_vs_neural_weight_scatter.png')))

## What Changed, And The Trade-Off

1. **Regularisation fixed the major overfitting failure.** The neural strategy no longer collapses out-of-sample. The probability gate, no-flip constraint, fewer features, and validation early stopping stop it from taking weak or directionally stale trades.

2. **It sizes down accepted trades.** Both strategies now trade the same gated rows, but neural has lower mean absolute weight and lower daily gross exposure. This improves Sharpe, but fixed earns more cumulative PnL because the accepted trades were profitable overall.

3. **The validation split is still small.** The training probabilities contain fewer than one thousand Energy rows, with uneven coverage across instruments. The validation block helps, but it is still a noisy basis for learning a nonlinear allocation rule.

4. **The model is a lightweight neural baseline, not a full TFT/VSN.** It uses a NumPy `tanh(x @ beta + ticker_bias) ** 2` allocation head. That follows the optional-session idea of optimizing a portfolio objective, but it is not the full lecture architecture with sequence modelling and variable selection.

5. **The fixed rule is already very strong in this period.** Fixed has high Sharpe because the primary signal plus probability confidence aligns well with Jan-Jun 2022. The neural overlay improves risk-adjusted performance only slightly, so the result should be presented as an incremental improvement, not a dramatic replacement.

## Practical Conclusion

For the current deliverable, the fixed volatility-targeted strategy remains the simpler and more explainable choice, with higher cumulative PnL. The regularised neural version is now a credible advanced comparison: it fixes the previous overfitting symptoms and slightly improves Sharpe, but the improvement is modest and comes from taking less risk rather than discovering a completely different profitable signal.